[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hoax12/fable-pytorch/blob/main/fable-pytorch/notebooks/08_memory_sequences.ipynb)

# ⚒️ Act II · Quest 08 — Memory — Sequences

*Order matters: recurrent networks that forecast signals and write text one character at a time.*

⬅️ [07_eyes_convolutions](07_eyes_convolutions.ipynb)  •  [09_attention_build_a_gpt](09_attention_build_a_gpt.ipynb) ➡️

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math, random

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {device}')
np.random.seed(0); random.seed(0)

# ══════════════════ ⚒️ FORGE GRADER — run me once ══════════════════
# Powers check() and xp_report(). Exercises give instant feedback + XP.
_XP = {"earned": 0, "done": set(), "checks": {}}

def _register(name, points, test, hint):
    _XP["checks"][name] = (points, test, hint)

def check(name, *answer):
    """Grade an exercise: check("ex1", your_answer). Repeatable until correct."""
    if name not in _XP["checks"]:
        print(f"unknown exercise: {name!r} — available: {list(_XP['checks'])}")
        return
    points, test, hint = _XP["checks"][name]
    try:
        ok = bool(test(*answer))
        err = None
    except Exception as e:
        ok, err = False, f"{type(e).__name__}: {e}"
    if ok:
        first = name not in _XP["done"]
        if first:
            _XP["done"].add(name)
            _XP["earned"] += points
        total = sum(p for p, _, _ in _XP["checks"].values())
        tag = f"+{points} XP" if first else "already earned"
        print(f"✅ {name} — correct! {tag}   ⚡ {_XP['earned']}/{total} XP")
    else:
        msg = f"❌ {name} — not yet."
        if err:
            msg += f" (your answer raised {err})"
        print(msg + f"\n   💡 hint: {hint}")

def xp_report():
    total = sum(p for p, _, _ in _XP["checks"].values())
    n = len(_XP["checks"])
    bar = "█" * int(10 * _XP["earned"] / max(total, 1)) or "░"
    print(f"⚡ XP: {_XP['earned']}/{total}   [{bar:<10}]   exercises: {len(_XP['done'])}/{n}")
    for name in _XP["checks"]:
        print(("  ✅ " if name in _XP["done"] else "  ⬜ ") + name)

## A new kind of input

Images arrive all at once. **Sequences** — sensor readings, text, audio — arrive in order, and
the *order is the information*. A recurrent network carries a hidden state `h`, its running
memory of everything seen so far:

\[ h_t = f(x_t, h_{t-1}) \]

`nn.GRU` and `nn.LSTM` add *gates* — learned valves controlling what to remember and forget.

### Part A — forecast a damped, drifting oscillation

In [ ]:
t = torch.linspace(0, 60, 1200)
signal = torch.exp(-t / 40) * torch.sin(t) + 0.02 * t + 0.05 * torch.randn_like(t)
plt.figure(figsize=(10, 2.5)); plt.plot(t, signal, lw=1)
plt.title("the signal: decaying oscillation + upward drift + noise"); plt.show()

# windows: last 30 points -> next point
W = 30
Xseq = torch.stack([signal[i:i + W] for i in range(len(signal) - W)]).unsqueeze(-1)
Yseq = signal[W:].unsqueeze(-1)
print("windows:", Xseq.shape, "targets:", Yseq.shape)

In [ ]:
class Forecaster(nn.Module):
    def __init__(self, hidden=48):
        super().__init__()
        self.gru = nn.GRU(1, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        out, h = self.gru(x)          # out: (B, T, hidden) — hidden state at EVERY step
        return self.head(out[:, -1])  # predict from the last step's memory

model = Forecaster().to(device)
opt = torch.optim.Adam(model.parameters(), lr=5e-3)
Xd, Yd = Xseq.to(device), Yseq.to(device)
for epoch in range(80):
    opt.zero_grad()
    loss = F.mse_loss(model(Xd), Yd)
    loss.backward(); opt.step()
print(f"final MSE: {loss.item():.5f}")

In [ ]:
# Roll the model forward on its OWN predictions (autoregressive)
model.eval()
window = signal[:W].tolist()
preds = []
with torch.no_grad():
    for _ in range(400):
        x = torch.tensor(window[-W:]).reshape(1, W, 1).to(device)
        nxt = model(x).item()
        preds.append(nxt); window.append(nxt)

plt.figure(figsize=(10, 2.8))
plt.plot(signal[:W + 400], label="truth", alpha=0.6)
plt.plot(range(W, W + 400), preds, "--", label="model, feeding itself")
plt.axvline(W, c="gray", ls=":"); plt.legend(); plt.title("autoregressive forecast"); plt.show()

Notice how errors slowly compound — each prediction becomes the next input. This *autoregressive*
loop is exactly how GPT generates text, one token at a time.

### Part B — a network that writes

Character-level language modeling: read characters, predict the **next** one. Our corpus:
tongue twisters (dense letter patterns, tiny vocabulary, fast to learn).

In [ ]:
corpus = (
    "she sells sea shells by the sea shore "
    "peter piper picked a peck of pickled peppers "
    "how much wood would a woodchuck chuck if a woodchuck could chuck wood "
    "fuzzy wuzzy was a bear fuzzy wuzzy had no hair "
) * 25

chars = sorted(set(corpus))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in corpus])
V = len(chars)
print(f"vocab: {V} chars | corpus: {len(data)} chars")

In [ ]:
SEQ = 32
def batch(bs=48):
    ix = torch.randint(0, len(data) - SEQ - 1, (bs,))
    xs = torch.stack([data[i:i + SEQ] for i in ix])
    ys = torch.stack([data[i + 1:i + SEQ + 1] for i in ix])   # shifted by one: next-char targets
    return xs.to(device), ys.to(device)

class Scribe(nn.Module):
    def __init__(self, V, emb=24, hidden=128):
        super().__init__()
        self.emb = nn.Embedding(V, emb)         # char id -> learned vector
        self.gru = nn.GRU(emb, hidden, batch_first=True)
        self.head = nn.Linear(hidden, V)
    def forward(self, x, h=None):
        out, h = self.gru(self.emb(x), h)
        return self.head(out), h

scribe = Scribe(V).to(device)
opt = torch.optim.Adam(scribe.parameters(), lr=3e-3)
for step in range(500):
    xs, ys = batch()
    logits, _ = scribe(xs)
    loss = F.cross_entropy(logits.reshape(-1, V), ys.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 125 == 0:
        print(f"step {step:4d}: loss {loss.item():.3f}")

In [ ]:
@torch.no_grad()
def write(prompt="she ", n=150, temperature=0.7):
    scribe.eval()
    idx = torch.tensor([[stoi.get(c, 0) for c in prompt]]).to(device)
    out = list(prompt)
    logits, h = scribe(idx)
    for _ in range(n):
        probs = F.softmax(logits[:, -1] / temperature, dim=-1)
        nxt = torch.multinomial(probs, 1)
        out.append(itos[nxt.item()])
        logits, h = scribe(nxt, h)
    return "".join(out)

for temp in [0.3, 0.8, 1.5]:
    print(f"--- temperature {temp} ---")
    print(write(temperature=temp), "\n")

**Temperature** divides the logits before softmax: low → confident and repetitive, high → chaotic.
The model learned spelling and word fragments from *nothing but next-char prediction* — the same
objective, scaled a billion-fold, produces ChatGPT. Next quest: the architecture that scaled.

In [ ]:
# ── this notebook's exercises (run once) ───────────────────────────────
_register("warmup", 5,
    lambda shp: tuple(shp) == (8, 15, 32),
    "nn.GRU(input_size=4, hidden_size=32, batch_first=True) on input (8, 15, 4): out.shape?")
_register("forecast_close", 15,
    lambda mse: mse < 0.003,
    "train the Forecaster longer (or bigger hidden) until MSE < 0.003; submit loss.item()")
_register("onehot", 15,
    lambda f: torch.equal(f(torch.tensor([1, 0, 2]), 4),
                          torch.tensor([[0., 1, 0, 0], [1., 0, 0, 0], [0., 0, 1, 0]])),
    "def onehot(idx, n): return F.one_hot(idx, n).float()  — or build with zeros + scatter")
_register("temp_quiz", 10,
    lambda s: "repet" in s.lower() or "safe" in s.lower() or "confident" in s.lower() or "determinist" in s.lower(),
    "one phrase: text at very LOW temperature becomes ______")

In [ ]:
gru = nn.GRU(input_size=4, hidden_size=32, batch_first=True)
out, h = gru(torch.randn(8, 15, 4))
check("warmup", out.shape)

## 🏆 Boss Challenges

Earn your XP — write each answer in a cell below and grade it with `check(...)`. When you're done, run `xp_report()`.

- `forecast_close` (15 XP) — get the forecaster MSE under `0.003`; submit the final loss value.
- `onehot` (15 XP) — write `onehot(idx, n)` producing float one-hot rows; submit the function.
- `temp_quiz` (10 XP) — what does very low temperature do to generated text?

In [ ]:
# ⚔️ your attempts here...

# xp_report()